# Chapter 7 — Scaling BPE tokenization
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/zerokaraLLM/blob/main/notebooks/ch07_tokenizer_scaling.ipynb)

A practical scaling exercise based on chapter 7. It trains a byte-level BPE vocabulary on a moderate corpus, serializes the merge table, encodes text, and benchmarks throughput. This chapter is CPU/memory bound rather than GPU bound.

In [ ]:
from collections import Counter
import json, time
corpus=('LLM systems need efficient tokenization. Byte pair encoding learns frequent byte sequences. ' * 500).encode()
print('bytes=',len(corpus))

## 1. Train a larger BPE merge table

In [ ]:
def counts(ids): return Counter(zip(ids,ids[1:]))
def merge(ids,pair,new_id):
    out=[]; i=0
    while i<len(ids):
        if i+1<len(ids) and (ids[i],ids[i+1])==pair: out.append(new_id); i+=2
        else: out.append(ids[i]); i+=1
    return out
def train(data,vocab_size=384):
    ids=list(data); merges={}
    for new_id in range(256,vocab_size):
        c=counts(ids)
        if not c: break
        pair=max(c,key=c.get); ids=merge(ids,pair,new_id); merges[pair]=new_id
    return merges
t=time.perf_counter(); merges=train(corpus,384); print('merges=',len(merges),'sec=',time.perf_counter()-t)

## 2. Serialize tokenizer state

In [ ]:
serial=[{'a':a,'b':b,'id':idx} for (a,b),idx in sorted(merges.items(),key=lambda kv:kv[1])]
with open('merges.json','w') as f: json.dump(serial,f)
print('saved merges.json')

## 3. Encode and benchmark

In [ ]:
def encode(text,merges):
    ids=list(text.encode())
    while True:
        candidates=[(merges[p],p) for p in counts(ids) if p in merges]
        if not candidates: break
        _,pair=min(candidates); ids=merge(ids,pair,merges[pair])
    return ids
sample='efficient tokenization for language models'
print(encode(sample,merges))
batch=[sample]*1000
t=time.perf_counter(); n=sum(len(encode(s,merges)) for s in batch); dt=time.perf_counter()-t
print('encoded tokens=',n,'sec=',dt,'texts/sec=',len(batch)/dt)

## Scaling note
Chapter 7 is about pushing tokenizer training/encoding toward a larger corpus. On Colab, increase corpus size gradually and watch RAM/CPU time. A T4 does not materially accelerate this pure-Python implementation.

Upstream: https://github.com/oreilly-japan/deep-learning-from-scratch-6/tree/main/ch07